In [1]:
import numpy as np

In [2]:
# generate a sequence with some noise
def generate_sequence(length,noise=0.1):
    return np.array([i+np.random.normal(0,noise) for i in range (length)])
#convert sequence into input/output pairs
def create_dataset(seq,n_stps):
    X,y=[],[]
    for i in range(len(seq)-n_stps):
        X.append(seq[i:i+n_stps])
        y.append(seq[i+n_stps])
    return np.array(X),np.array(y)

In [3]:
#prepare data
np.random.seed(42)
sequence=generate_sequence(100,noise=0.5)

In [4]:
n_steps=3
X,y=create_dataset(sequence,n_steps)
X=X.reshape((X.shape[0],X.shape[1],1))

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import SimpleRNN, Dense

In [6]:
#Build model with regularization and grddient clipping from the start
# Build model with regularization and gradient clipping from the start
model = Sequential([
    SimpleRNN(
        50, activation = 'relu', input_shape = (n_steps, 1),
        kernel_regularizer = l2(0.01), recurrent_regularizer = l2(0.01)
    ),
    Dense(1)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
# Compile with gradient clipped optimizer
optimizer = tf.keras.optimizers.Adam(clipvalue = 0.1)
model.compile(optimizer = optimizer, loss = 'mse')

In [8]:
# Train
model.fit(X,y, epochs = 100, verbose = 0)

# Predict
x_input = np.array([97, 98, 99])
x_input = x_input.reshape((1, n_steps, 1))
yhat = model.predict(x_input, verbose = 0)
print(f"Predicted next value: {yhat[0][0]:.4f}")
print(f"Actual  next value: ~100")

Predicted next value: 102.2776
Actual  next value: ~100
